In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix,
                             classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../dataset/combined_features.csv')
print("Shape:", df.shape)
df.head()

Shape: (10000, 43)


,url_length,domain_length,path_length,num_dots,num_hyphens,num_slashes,num_at,num_digits,num_params,num_percent,...,num_forms,num_inputs,has_password_field,num_external_scripts,num_external_images,num_hidden_elements,page_size,fetch_status,url,label
0,14,13,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,thecanadianencyclopedia.com/index.cfm?PgNm=TCE...,0
1,56,23,33,3,1,5,0,5,0,0,...,0,0,0,11,1,0,77418,1,soundcloud.com/a-trak/robyn-indestructible-a-t...,0
2,146,19,127,4,13,7,0,8,0,0,...,0,0,0,0,0,0,1530,1,linkedin.com/pub/peter-wege/12/336/926,0
3,45,9,36,1,3,3,0,1,0,0,...,0,0,0,0,0,0,9805,1,chacha.com/question/how-much-are-charles-barkl...,0
4,56,27,29,3,1,3,0,6,0,0,...,1,2,0,1,1,0,195061,1,toronto-theatre.com/theaters/bluma-appel-theat...,0


In [2]:
# Drop non-feature columns
X = df.drop(['url', 'label', 'fetch_status'], axis=1)
y = df['label']

print("Features shape:", X.shape)
print("Label distribution:\n", y.value_counts())

Features shape: (10000, 40)
Label distribution:
 label
0    5000
1    5000
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining label distribution:\n", y_train.value_counts())
print("Testing label distribution:\n", y_test.value_counts())

Training set: (8000, 40)
Testing set: (2000, 40)

Training label distribution:
 label
0    4000
1    4000
Name: count, dtype: int64
Testing label distribution:
 label
0    1000
1    1000
Name: count, dtype: int64


In [4]:
print("Training model...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("Training complete!")

Training model...
Training complete!


In [5]:
y_pred = rf_model.predict(X_test)

print("=== Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))

=== Model Performance ===
Accuracy:  0.6675
Precision: 0.6541
Recall:    0.7110
F1 Score:  0.6814

Classification Report:
              precision    recall  f1-score   support

      Benign       0.68      0.62      0.65      1000
   Malicious       0.65      0.71      0.68      1000

    accuracy                           0.67      2000
   macro avg       0.67      0.67      0.67      2000
weighted avg       0.67      0.67      0.67      2000



In [6]:
# Load the full URL features dataset
df_url_full = pd.read_csv('../dataset/url_features.csv')

X_full = df_url_full.drop('label', axis=1)
y_full = df_url_full['label']

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print("Training URL-only model on full dataset...")
rf_url_only = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_url_only.fit(X_train_full, y_train_full)

y_pred_full = rf_url_only.predict(X_test_full)

print("=== URL-Only Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test_full, y_pred_full):.4f}")
print(f"Precision: {precision_score(y_test_full, y_pred_full):.4f}")
print(f"Recall:    {recall_score(y_test_full, y_pred_full):.4f}")
print(f"F1 Score:  {f1_score(y_test_full, y_pred_full):.4f}")

Training URL-only model on full dataset...
=== URL-Only Model Performance ===
Accuracy:  0.9016
Precision: 0.9013
Recall:    0.9020
F1 Score:  0.9016


In [7]:
# Filter only successfully fetched URLs
df_fetched = df[df['fetch_status'] == 1]

X_fetched = df_fetched.drop(['url', 'label', 'fetch_status'], axis=1)
y_fetched = df_fetched['label']

print("Successfully fetched samples:", len(df_fetched))
print("Label distribution:\n", y_fetched.value_counts())

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fetched, y_fetched, test_size=0.2, random_state=42, stratify=y_fetched
)

print("\nTraining improved hybrid model...")
rf_hybrid = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_hybrid.fit(X_train_f, y_train_f)

y_pred_f = rf_hybrid.predict(X_test_f)

print("\n=== Improved Hybrid Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test_f, y_pred_f):.4f}")
print(f"Precision: {precision_score(y_test_f, y_pred_f):.4f}")
print(f"Recall:    {recall_score(y_test_f, y_pred_f):.4f}")
print(f"F1 Score:  {f1_score(y_test_f, y_pred_f):.4f}")

Successfully fetched samples: 6407
Label distribution:
 label
0    3718
1    2689
Name: count, dtype: int64

Training improved hybrid model...

=== Improved Hybrid Model Performance ===
Accuracy:  0.7254
Precision: 0.6906
Recall:    0.6264
F1 Score:  0.6569


In [8]:
from sklearn.utils import resample

# Filter only successfully fetched URLs
df_fetched = df[df['fetch_status'] == 1]

# Balance the fetched dataset
benign_fetched = df_fetched[df_fetched['label'] == 0]
malicious_fetched = df_fetched[df_fetched['label'] == 1]

# Undersample benign to match malicious count
benign_resampled = resample(benign_fetched,
                             replace=False,
                             n_samples=len(malicious_fetched),
                             random_state=42)

df_balanced_fetched = pd.concat([benign_resampled, malicious_fetched])
df_balanced_fetched = df_balanced_fetched.sample(frac=1, random_state=42).reset_index(drop=True)

X_bal = df_balanced_fetched.drop(['url', 'label', 'fetch_status'], axis=1)
y_bal = df_balanced_fetched['label']

print("Balanced fetched samples:", len(df_balanced_fetched))
print("Label distribution:\n", y_bal.value_counts())

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

print("\nTraining balanced hybrid model...")
rf_balanced = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_balanced.fit(X_train_b, y_train_b)

y_pred_b = rf_balanced.predict(X_test_b)

print("\n=== Balanced Hybrid Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test_b, y_pred_b):.4f}")
print(f"Precision: {precision_score(y_test_b, y_pred_b):.4f}")
print(f"Recall:    {recall_score(y_test_b, y_pred_b):.4f}")
print(f"F1 Score:  {f1_score(y_test_b, y_pred_b):.4f}")

Balanced fetched samples: 5378
Label distribution:
 label
1    2689
0    2689
Name: count, dtype: int64

Training balanced hybrid model...

=== Balanced Hybrid Model Performance ===
Accuracy:  0.7305
Precision: 0.7222
Recall:    0.7491
F1 Score:  0.7354


In [9]:
# Save both models
joblib.dump(rf_url_only, '../model/url_only_model.pkl')
joblib.dump(rf_balanced, '../model/hybrid_model.pkl')

print("Both models saved successfully!")

Both models saved successfully!


In [10]:
# Verify URL-only model works correctly
test_url = extract_url_features('http://google.com') if False else None

# Check what rf_url_only predicts on a benign sample
test_sample = X_test_full[y_test_full == 0].iloc[0:1]
pred = rf_url_only.predict(test_sample)[0]
print("Should be 0 (benign):", pred)

# Resave explicitly
import joblib
joblib.dump(rf_url_only, '../model/url_only_model.pkl')
joblib.dump(rf_balanced, '../model/hybrid_model.pkl')
print("Models resaved correctly!")

Should be 0 (benign): 1
Models resaved correctly!


In [11]:
df_orig = pd.read_csv('../dataset/balanced_links.csv')
short_orig = df_orig[df_orig['url'].str.len() <= 12]
print("\nShort URL types:")
print(short_orig['type'].value_counts())
print("\nSample short URLs:")
print(short_orig[['url', 'type']].head(10).to_string())


Short URL types:
type
phishing    2500
benign      2151
malware      161
Name: count, dtype: int64

Sample short URLs:
               url      type
125             ºE  phishing
241    jmvwiby.tk/    benign
418   sherdog.com/    benign
468       hnu.edu/    benign
531   sovpark.com/    benign
594   cfldaily.ca/    benign
683   ablabels.com  phishing
685    rujva.co.uk  phishing
781    btwvmca.tk/    benign
1000  0rz.tw/oot8G    benign
